In [ ]:
import requests
import time
import numpy as np 
import matplotlib.pyplot as plt
import pandas as pd 
import datetime
import requests
import time
import pickle
import json
import os
import copy

In [ ]:

# 1695 = Open World
# 1662 = Survival
# 3810 = Sandbox
# 1100689 = Open World Survival Craft
tag_ids = [1695, 1662, 3810, 1100689]
id_set = set()
game_list = [list(), list(), list(), list()]
y = 0
tag_combinations = set()
for tag_id in tag_ids:
    request_tring = 'https://api.steampowered.com/IStoreQueryService/Query/v1/?input_json={"query":{"filters":{"type_filters":{"include_games":true},"tagids_must_match":[{"tagids":["'
    request_tring += str(tag_id)
    request_tring += '"]}]}},"context":{"language":"english","country_code":"US","steam_realm":"1"},"data_request":{"include_basic_info":""}}'
    request = requests.get(request_tring)
    data = request.json()
    response = data['response']
    count = response['metadata']['total_matching_records']
    #print(count)
    time.sleep(0.5) 
    for i in range(0,count,100):
        #print(i)
        #'{"query":{"start":"10","count":"10","filters":{"type_filters":{"include_games":true},"tagids_must_match":[{"tagids":["1100689"]}]}},"context":{"language":"english","country_code":"US"},"data_request":{"include_basic_info":""}}'
        request_tring = 'https://api.steampowered.com/IStoreQueryService/Query/v1/?input_json={"query":{"start":"'
        request_tring += str(i)
        request_tring += '","count":"100","filters":{"type_filters":{"include_games":true},"tagids_must_match":[{"tagids":["'
        request_tring += str(tag_id)
        request_tring += '"]}]}},"context":{"language":"english","country_code":"US","steam_realm":"1"},' \
        '"data_request":{"include_basic_info":false,"include_reviews":true,"include_ratings":true,"include_release":true}}'
        request = requests.get(request_tring)
        data = request.json()
        response = data['response']
        ids = response['ids']
        games = response['store_items']
        for game_id in ids:
            id_set.add(int(game_id['appid']))
        for game in games:
            game_list[y].append(game)
        time.sleep(0.5) 
        if i%100 == 0:
            print(tag_id, i, "/", count)
    y+=1

In [ ]:
pickle.dump(game_list, open("game_list", 'wb'))

In [2]:
array = pickle.load(open("game_list", 'rb')) 

In [3]:
id_0 = set()
for item in array[0]:
    id_0.add(item["id"])
id_1 = set()
for item in array[1]:
    id_1.add(item["id"])
id_2 = set()
for item in array[2]:
    id_2.add(item["id"])
id_3 = set()
for item in array[3]:
    id_3.add(item["id"])

combined_ids_0_0 = id_0 & id_1  & id_2
combined_ids_0_1 = id_0 & id_1 
combined_ids_1_2 = id_1 & id_2
combined_ids_2_0 = id_0 & id_2
combined_ids = combined_ids_0_1 | combined_ids_1_2 | combined_ids_2_0 | id_3
print(len(combined_ids), len(combined_ids_0_0))


2824 381


In [4]:
testgames = array[0] + array[1] +  array[2] + array[3]

In [7]:
allready_done = set()
games = []
for filename in os.listdir("./outputs/"):
    if filename.startswith("review_") and not filename.endswith("texts.pk"):
        with open(f"./outputs/{filename}", "rb") as f:
            try:
                while True:
                    results = pickle.load(f)
                    allready_done.add(results[0])
                    #games.append(results)
                        #print(r)
            except EOFError:
                pass
print(len(allready_done))

0


In [8]:
cursor = "*"
#allready_done = set()
all_games = list()
languages = ["german", "english"]
import urllib.parse
#safe_string = urllib.parse.quote_plus(...)
for i in range(0, len(testgames)):
    abnormal_exit = False 
    review_text_dict = dict()
    review_text_dict_filtered = dict()
    # 'reviews': {'summary_filtered': {'review_count': 20,
    if "reviews" in testgames[i].keys():
        if "summary_filtered" in testgames[i]["reviews"].keys():
            if "review_count" in testgames[i]["reviews"]["summary_filtered"].keys():
             if testgames[i]["reviews"]["summary_filtered"]["review_count"] <= 100:
                 continue
    cursor = "*"
    game_reviews = list()
    game_reviews_with_bombing = list()
    game_id = testgames[i]["id"]

    
    
    if game_id in combined_ids and game_id not in allready_done:
        try:
            allready_done.add(game_id)
            
            game_name = testgames[i]["name"]
    
    
    
            therearenofaces = True
            reviewcounts_fromfirst = 0
            reviewcounts_fromsecond = 0
            requeststring = f"https://store.steampowered.com/appreviews/{game_id}?json=1&language=all&purchase_type=all&filter_offtopic_activity=1&filter=recent&num_per_page=100&cursor={cursor}"
            request = requests.get(requeststring)
            data = request.json()
            #print(requeststring)
            if "query_summary" in data.keys():
                if "total_reviews" in data["query_summary"].keys():
                    reviewcounts_fromfirst = data["query_summary"]["total_reviews"]
            requeststring = f"https://store.steampowered.com/appreviews/{game_id}?json=1&language=all&purchase_type=all&filter_offtopic_activity=0&filter=recent&num_per_page=100&cursor={cursor}"
            request = requests.get(requeststring)
            data = request.json()
            #print(requeststring)
            if "query_summary" in data.keys():
                if "total_reviews" in data["query_summary"].keys():
                    reviewcounts_fromsecond = data["query_summary"]["total_reviews"]
        
            if reviewcounts_fromfirst == reviewcounts_fromsecond:
                therearenofaces = False
            
            print(f"Next Game: {game_name} - {game_id} Reviewcounts: {reviewcounts_fromfirst} - {reviewcounts_fromsecond} Difference: {reviewcounts_fromfirst - reviewcounts_fromsecond}") 
            
            release_date = 0
            id_list_no_filter = []
            id_list_with_filter = []
            if "release" in testgames[i].keys():
                if "steam_release_date" in testgames[i]["release"].keys():
                    release_date = datetime.datetime.fromtimestamp(int(testgames[i]["release"]["steam_release_date"]), datetime.UTC).strftime('%m-%d-%Y')
            try:
                for j in range(0, reviewcounts_fromfirst, 100):
                    requeststring = f"https://store.steampowered.com/appreviews/{game_id}?json=1&language=all&purchase_type=all&filter_offtopic_activity=1&filter=recent&num_per_page=100&cursor={cursor}"
                    request = requests.get(requeststring)
                    data = request.json()
                    #print(requeststring)
                    #print(data)
                    #break
                    cursor = urllib.parse.quote_plus(data["cursor"])
                    reviews_in_data = data["reviews"]
                    for review in reviews_in_data:
                        datum = datetime.datetime.fromtimestamp(int(review["timestamp_created"]), datetime.UTC).strftime('%m-%d-%Y')
                        review_id = review["recommendationid"] #recommendationid
                        id_list_no_filter.append(review_id)
                        positive = 0
                        negative = -1
                        if review["voted_up"]:
                            positive = 1
                            negative = 0
                        review_dict = {"positive":positive, "negative":negative, "created":datum, "review_id":review_id}
                        if "written_during_early_access" in review.keys():
                            review_dict["written_during_early_access"] = review["written_during_early_access"] 
                        if "review" in review.keys() and review["language"] in languages and therearenofaces:
                            review_text_dict[review_id] = review["review"] 
                        game_reviews.append(review_dict)
                    #print(review_text_dict)
                    #break
            except:
                print(f"{game_id} - {game_name} crashed during first")
                abnormal_exit = True 
            cursor = "*"
            if not therearenofaces:
                print("Copying")
                game_reviews_with_bombing = game_reviews
                id_list_with_filter = id_list_no_filter
            else:
                try:
                    for j in range(0, reviewcounts_fromsecond, 100):
                        requeststring = f"https://store.steampowered.com/appreviews/{game_id}?json=1&language=all&purchase_type=all&filter_offtopic_activity=0&filter=recent&num_per_page=100&cursor={cursor}"
                        request = requests.get(requeststring)
                        data = request.json()
                        #print(requeststring)
                        #print(data)
                        #break
                        cursor = urllib.parse.quote_plus(data["cursor"])
                        reviews_in_data = data["reviews"]
                        for review in reviews_in_data:
                            datum = datetime.datetime.fromtimestamp(int(review["timestamp_created"]), datetime.UTC).strftime('%m-%d-%Y')
                            review_id = review["recommendationid"] #recommendationid
                            id_list_with_filter.append(review_id)
                            # datetime.datetime.fromtimestamp(int(review["timestamp_created"]), datetime.UTC)
                            positive = 0
                            negative = -1
                            if review["voted_up"]:
                                positive = 1
                                negative = 0
                            review_dict = {"positive":positive, "negative":negative, "created":datum, "review_id":review_id}
                            if "written_during_early_access" in review.keys():
                                review_dict["written_during_early_access"] = review["written_during_early_access"] 
                            if "review" in review.keys() and review["language"] in languages:
                                review_text_dict[review_id] = review["review"] 
                            game_reviews_with_bombing.append(review_dict)
                        #print(review_text_dict)
                        #break
                except:
                    print(f"{game_id} - {game_name} crashed during second")
                    abnormal_exit = True 
            print(game_id, game_name, len(game_reviews), len(game_reviews_with_bombing), len(game_reviews)-len(game_reviews_with_bombing))
            if len(game_reviews)>0 and len(game_reviews_with_bombing) >0:
                diff_review_ids = set(id_list_with_filter) ^ set(id_list_no_filter)
                entry_to_export = [game_id, game_name, release_date, pd.DataFrame(game_reviews).groupby('created', as_index=False).agg(
                    {"positive": "sum","negative": "sum", "review_id": list,"written_during_early_access": "sum"}
                ), pd.DataFrame(game_reviews_with_bombing).groupby('created', as_index=False).agg(
                    {"positive": "sum","negative": "sum", "review_id": list,"written_during_early_access": "sum"}
                ), diff_review_ids, abnormal_exit]
                  
                pickle.dump(entry_to_export, open(f"outputs/review_{game_id}.pk", 'wb'))
                for review_id in diff_review_ids:
                    if review_id in review_text_dict.keys() and not abnormal_exit:
                        review_text_dict_filtered[review_id] = review_text_dict[review_id]
                if len(diff_review_ids) > 0 and not abnormal_exit:
                    #print (review_text_dict_filtered)
                    pickle.dump(review_text_dict_filtered, open(f"outputs/text_{game_id}.pk", 'wb'))
        except:
            print(f"{ testgames[i]["name"]} - {game_id} crashed on first webcall")
        #break
print("Done")

# 648800 - Raft crashed during second
# 105600 - Terraria crashed during first

Next Game: Terra Randoma - 1120400 Reviewcounts: 163 - 163 Difference: 0
Copying
1120400 Terra Randoma 163 163 0
Next Game: They Are Hundreds - 806860 Reviewcounts: 186 - 186 Difference: 0
Copying
806860 They Are Hundreds 186 186 0
Next Game: Wild Wolf - 749540 Reviewcounts: 147 - 147 Difference: 0
Copying
749540 Wild Wolf 147 147 0
Next Game: 大逃亡专家 Escape Expert - 727180 Reviewcounts: 142 - 142 Difference: 0
Copying
727180 大逃亡专家 Escape Expert 142 142 0
Next Game: Play With Gilbert - Remake - 740040 Reviewcounts: 138 - 138 Difference: 0
Copying
740040 Play With Gilbert - Remake 138 138 0
Next Game: PostCollapse - 509770 Reviewcounts: 133 - 133 Difference: 0
Copying
509770 PostCollapse 133 133 0
Next Game: CrossWorlds: Escape - 511430 Reviewcounts: 158 - 158 Difference: 0
Copying
511430 CrossWorlds: Escape 158 158 0
Next Game: Badiya: Desert Survival - 545050 Reviewcounts: 342 - 342 Difference: 0
Copying
545050 Badiya: Desert Survival 342 342 0
Next Game: Last Holiday - 2115850 Reviewco